In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import multinomial
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from PIL import Image

In [ ]:
# Create custom seaborn style
custom_style = {
    'axes.facecolor': '#9fc6c7',
    'figure.facecolor': '#86a2a3',
    'grid.linestyle': ':',
    'grid.linewidth': 0.5,
    'axes.linewidth': 0.5,
    'grid.color': 'white',
    
    # Font settings - make text solid and dark
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'font.family': 'Calibri', 

    # **Fix transparency/color issues**
    'text.color': 'black',           # Make all text solid black
    'axes.labelcolor': 'black',      # Axis labels black
    'xtick.color': 'black',          # Tick marks black
    'ytick.color': 'black',          # Tick marks black
    'axes.edgecolor': 'black',       # Axes spines black
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    
}

print_rep = 50

sns.set_style('darkgrid', rc=custom_style)

#### Categorical (Multinoulli) Distribution

<table align="left">
  <tr>
    <th align="left"><b>Concept</b></th>
    <th align="left"><b>Bernoulli</b></th>
    <th align="left"><b>Categorical</b></th>
  </tr>
  <tr>
    <td align="left"><b>Number of outcomes</b></td>
    <td align="left">2 (success/failure)</td>
    <td align="left"><i>K</i> (3 or more categories)</td>
  </tr>
  <tr>
    <td align="left"><b>Parameters</b></td>
    <td align="left"><i>p</i> (success probability)</td>
    <td align="left"><i>p</i><sub>1</sub>, <i>p</i><sub>2</sub>, ..., <i>p</i><sub>K</sub> (sum to 1)</td>
  </tr>
  <tr>
    <td align="left"><b>Example</b></td>
    <td align="left">Click/No-click</td>
    <td align="left">Dog/Cat/Bird</td>
  </tr>
  <tr>
    <td align="left"><b>ML connection</b></td>
    <td align="left">Binary classification</td>
    <td align="left">Multi-class classification</td>
  </tr>
  <tr>
    <td align="left"><b>Output activation</b></td>
    <td align="left">Sigmoid (1 neuron)</td>
    <td align="left">Softmax (K neurons)</td>
  </tr>
</table>

In [ ]:
# 3 categories: Dog, Cat, Bird with probabilities
probs = [0.5, 0.3, 0.2]  # 50% dog, 30% cat, 20% bird
categories = ['Dog', 'Cat', 'Bird']

# 1000 draws
samples = np.random.choice(categories, size=1000, p=probs)

# Count occurrences
unique, counts = np.unique(samples, return_counts=True)
for cat, count in zip(unique, counts):
    print(f"{cat}: {count/1000:.1%}")

# Plot
plt.bar(categories, counts/1000)
plt.axhline(y=probs[0], color='r', linestyle='--', label=f'Dog true: {probs[0]}')
plt.axhline(y=probs[1], color='g', linestyle='--', label=f'Cat true: {probs[1]}')
plt.axhline(y=probs[2], color='b', linestyle='--', label=f'Bird true: {probs[2]}')
plt.title('Categorical Distribution (1000 samples)')
plt.ylabel('Proportion')
plt.legend()
plt.show()

# Using scipy
rv = multinomial(n=1, p=probs)  # n=1 = single trial
print(f"PMF for Dog: {rv.pmf([1,0,0])}")
print(f"PMF for Cat: {rv.pmf([0,1,0])}")
print(f"PMF for Bird: {rv.pmf([0,0,1])}")

#### The ML Connection: Softmax & Cross-Entropy

Just as Bernoulli connects to sigmoid and binary cross-entropy, Categorical connects to softmax and categorical cross-entropy.

<table align="left">
  <tr>
    <th align="left"><b>Application</b></th>
    <th align="left"><b>Categories</b></th>
    <th align="left"><b>How Categorical is Used</b></th>
  </tr>
  <tr>
    <td align="left">Image classification</td>
    <td align="left">Dog, Cat, Bird, Car, ...</td>
    <td align="left">Model outputs probability for each class</td>
  </tr>
  <tr>
    <td align="left">Sentiment analysis</td>
    <td align="left">Positive, Negative, Neutral</td>
    <td align="left">Softmax converts logits to probabilities</td>
  </tr>
  <tr>
    <td align="left">Recommendation</td>
    <td align="left">Click on item A/B/C/D</td>
    <td align="left">Predict probability user clicks each item</td>
  </tr>
  <tr>
    <td align="left">Document classification</td>
    <td align="left">Sports, Politics, Tech, Health</td>
    <td align="left">Each document assigned one category</td>
  </tr>
  <tr>
    <td align="left">Customer segmentation</td>
    <td align="left">Premium, Standard, Basic</td>
    <td align="left">Predict segment probability</td>
  </tr>
</table>

In [ ]:
'''
sklearn docs:
‘lbfgs’ is a good default solver because it works reasonably well for a wide class of problems.
For multiclass problems (n_classes >= 3), all solvers except ‘liblinear’ minimize the full multinomial loss, ‘liblinear’ will raise an error.
'''

# A simple MNIST classifier
digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(solver='lbfgs', max_iter=1000)
model.fit(X_train, y_train)

# Predict probabilities (categorical distribution)
probs = model.predict_proba(X_test)  # Shape: (n_samples, 10)

# For first test sample, see the categorical distribution
print(f"Probabilities for first test image: {probs[0]}")
print(f"Predicted class: {np.argmax(probs[0])}")
print(f"True class: {y_test[0]}")
print(f"Sum of probabilities: {probs[0].sum():.3f}")  # 1.0

# Evaluate
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
loss = log_loss(y_test, probs)

print(f"Accuracy: {acc:.2%}")
print(f"Categorical cross-entropy loss: {loss:.3f}")

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
# be careful

model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
# The parameter will be removed entirely. The model will always use 'multinomial' for multi-class problems

#### The One-Hot Encoding

In [ ]:
# 3 classes: Dog, Cat, Bird
# Dog = [1, 0, 0]
# Cat = [0, 1, 0]
# Bird = [0, 0, 1]

# In ML, the output is a probability vector:
# [0.5, 0.3, 0.2] → 50% Dog, 30% Cat, 20% Bird

# The loss function compares these two vectors
y_true = np.array([0, 1, 0])  # Cat
y_pred = np.array([0.5, 0.3, 0.2])  # Model prediction

# Categorical cross-entropy = -sum(y_true * log(y_pred))
loss = -np.sum(y_true * np.log(y_pred))
print(f"Loss for Cat example: {loss:.3f}")
# Only the element at index 1 contributes to loss
# = -log(0.3) = 1.204

#### Multi-Label vs Multi-Class (Important Distinction!)

https://scikit-learn.org/stable/modules/multiclass.html

In [ ]:
img = Image.open('../img/multi_org_chart.png')
aspect = 1
img.resize((int(img.size[0] * aspect), int(img.size[1] * aspect)))

Don't confuse Categorical (multi-class) with multi-label

<table align="left">
  <tr>
    <th align="left"><b>Aspect</b></th>
    <th align="left"><b>Multi-Class (Categorical)</b></th>
    <th align="left"><b>Multi-Label (Multiple Bernoulli)</b></th>
  </tr>
  <tr>
    <td align="left"><b>Assignment</b></td>
    <td align="left">Exactly ONE class</td>
    <td align="left">Any number of classes can apply</td>
  </tr>
  <tr>
    <td align="left"><b>Example</b></td>
    <td align="left">Animal type: Dog, Cat, Bird</td>
    <td align="left">Image tags: Dog, Cat, Bird</td>
  </tr>
  <tr>
    <td align="left"><b>Output</b></td>
    <td align="left">Softmax (sums to 1)</td>
    <td align="left">Multiple sigmoids (independent)</td>
  </tr>
  <tr>
    <td align="left"><b>Loss</b></td>
    <td align="left">Categorical cross-entropy</td>
    <td align="left">Binary cross-entropy per label</td>
  </tr>
  <tr>
    <td align="left"><b>Distribution</b></td>
    <td align="left">Categorical</td>
    <td align="left">Product of Bernoullis</td>
  </tr>
</table>